In [2]:
import torch
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
from huggingface_hub import login
login()

quantization_config = BitsAndBytesConfig(
    load_in_4bit = True, bnb_4bit_compute_dtype=torch.float16
)
model_name = 'meta-llama/Llama-3.1-8B'
tokenizer = AutoTokenizer.from_pretrained(model_name,quantization_config=quantization_config, device_map = 'auto')
model = AutoModel.from_pretrained(model_name, dtype = torch.float16)

# get the embedding matrix
embeddings = (model.embed_tokens.weight if hasattr(model, 'embed_tokens') else model.wte.weight) # shape -> [vocab_size, hidden_dim]
print(f'Embedding matrix shape: {embeddings.shape}')

def get_token_embedding(word):
  # get the embedding for a word (first token if multitoken)
  token_ids = tokenizer.encode(word)
  token_id = token_ids[0]
  token_text = tokenizer.decode([token_id])
  embedding = embeddings[token_id].float()
  return embedding, token_id, token_text

words = ["Python", "JavaScript", "Java", "snake", "coffee", "espresso", "tea"]
word_embeddings = {}

print("\n" + "=" * 50)
print("TOKEN EMBEDDINGS (Llama 3.1 8B)")
print("=" * 50)
for word in words:
    emb, tid, text = get_token_embedding(word)
    word_embeddings[word] = emb
    print(f"{word:12} -> token {tid:6} '{text}' -> [{emb[0]:.3f}, {emb[1]:.3f}, ...]")

def cosine_similarity(a, b):
    return torch.dot(a, b) / (torch.norm(a) * torch.norm(b))

# compute cosine similarities
print("Cosine Similarity")
pairs = [("Python", "JavaScript"), ("coffee", "espresso"),
         ("Python", "snake"), ("Java", "coffee")]
for w1, w2 in pairs:
  sim = cosine_similarity(word_embeddings[w1],word_embeddings[w2])
  print(f"sim{w1:12}, {w2:12} = {sim:.4f}")

print("\nExpected: Python-JavaScript > Python-snake")
print("Java-coffee is interesting: programming language vs the drink!")

# ============================================================
# EMBEDDING MATRIX STATS
# ============================================================
print(f"\nVocabulary size:     {embeddings.shape[0]:,}")
print(f"Embedding dimension: {embeddings.shape[1]:,}")
print(f"Total parameters:    {embeddings.numel():,}")
print(f"Memory (FP16):       {embeddings.numel() * 2 / 1e6:.1f} MB")
print(f"Memory (FP32):       {embeddings.numel() * 4 / 1e6:.1f} MB")



config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

LlamaModel LOAD REPORT from: meta-llama/Llama-3.1-8B
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding matrix shape: torch.Size([128256, 4096])

TOKEN EMBEDDINGS (Llama 3.1 8B)
Python       -> token 128000 '<|begin_of_text|>' -> [0.001, -0.001, ...]
JavaScript   -> token 128000 '<|begin_of_text|>' -> [0.001, -0.001, ...]
Java         -> token 128000 '<|begin_of_text|>' -> [0.001, -0.001, ...]
snake        -> token 128000 '<|begin_of_text|>' -> [0.001, -0.001, ...]
coffee       -> token 128000 '<|begin_of_text|>' -> [0.001, -0.001, ...]
espresso     -> token 128000 '<|begin_of_text|>' -> [0.001, -0.001, ...]
tea          -> token 128000 '<|begin_of_text|>' -> [0.001, -0.001, ...]
Cosine Similarity
simPython      , JavaScript   = 1.0000
simcoffee      , espresso     = 1.0000
simPython      , snake        = 1.0000
simJava        , coffee       = 1.0000

Expected: Python-JavaScript > Python-snake
Java-coffee is interesting: programming language vs the drink!

Vocabulary size:     128,256
Embedding dimension: 4,096
Total parameters:    525,336,576
Memory (FP16):       1050.7 MB
Mem